In [ ]:
# https://www.philschmid.de/optimize-sentence-transformers
### https://huggingface.co/docs/transformers/v4.21.0/en/add_new_pipeline
### https://www.philschmid.de/convert-transformers-to-onnx

In [ ]:
model_type_str = "tohoku-bbwwm_vocab-added_jppost-cp-e1-b512_jsts-100-cp-e3-b1"
model_name_str = "../../fine-tuning/tuned_models/tohoku-bbwwm_vocab-added_jppost-cp-e1-b512_jsts-100-cp-e1to5-b1/checkpoints/37353"
# model_name_str = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

max_length_token_int = 512 # 128 or 512

dataset_name = 'mixed'
# path_dataset_str = '../../datasets/jppost/mixed/data_sbert_speed_900.csv'
path_dataset_str = '../../datasets/jppost/mixed/data_sbert_speed_700.csv'
# path_dataset_str = '../../datasets/jppost/mixed/data_sbert_speed_200.csv'

y_max_float = 0.180


print(f"{model_type_str = }")
print(f"{model_name_str = }")
print(f"{max_length_token_int = }")
print(f"{dataset_name = }")
print(f"{path_dataset_str = }")
print(f"{y_max_float = }")


In [ ]:
# define model type names and paths
filepath_output_eval_jsts_str = "scores_jsts_pearson-spearman/scores_jsts_" + model_type_str + ".txt"

modeltype_onnx_str = model_type_str + "_onnx-optim"
modeltype_optd_str = model_type_str + "_onnx-optim-optd"
modeltype_qtzd_str = model_type_str + "_onnx-optim-qtzd"

modelpath_output_onnx_str = "onnxed_models/" + modeltype_onnx_str
modelpath_output_optd_str = "onnxed_models/" + modeltype_optd_str
modelpath_output_qtzd_str = "onnxed_models/" + modeltype_qtzd_str


print(f"{filepath_output_eval_jsts_str = }")
print(f"{modeltype_onnx_str = }")
print(f"{modeltype_optd_str = }")
print(f"{modeltype_qtzd_str = }")
print(f"{modelpath_output_onnx_str = }")
print(f"{modelpath_output_optd_str = }")
print(f"{modelpath_output_qtzd_str = }")


In [ ]:
### Define Pipeline
from transformers import Pipeline
import torch

# copied from the model card
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


class SentenceTransformerPipeline(Pipeline):
    def _sanitize_parameters(self, **kwargs):
        # we don't have any hyperameters to sanitize
        preprocess_kwargs = {}
        return preprocess_kwargs, {}, {}

    def preprocess(self, inputs):
        encoded_inputs = self.tokenizer(
            inputs,
            padding = True,
            truncation = True,
            max_length = max_length_token_int,
            return_tensors='pt'
        )
        return encoded_inputs

    def _forward(self, model_inputs):
        outputs = self.model(**model_inputs)
        return {"outputs": outputs, "attention_mask": model_inputs["attention_mask"]}

    def postprocess(self, model_outputs):
        # Perform pooling
        sentence_embeddings = mean_pooling(model_outputs["outputs"], model_outputs['attention_mask'])
        # Normalize embeddings
        # sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
        return sentence_embeddings


In [ ]:
### Define a function for evaluating models
from sentence_transformers import util
from scipy.stats import pearsonr, spearmanr


def evaluate_model(st_pipeline, data_eval_dict, type_str, filepath_output_str):
    embeddings1_list = st_pipeline(data_eval_dict[sentence1_str])
    print("embeddings1 done")
    embeddings2_list = st_pipeline(data_eval_dict[sentence2_str])
    print("embeddings2 done")
    
    cossim_list = list()
    for i, (embedding1, embedding2) in enumerate(zip(embeddings1_list, embeddings2_list)):
        cossim_float = util.cos_sim(embedding1[0].tolist(), embedding2[0].tolist()).tolist()[0][0]
        # print(data_eval_dict[label_str][i] / 5, cossim_float)
        cossim_list.append(cossim_float)
    
    pearson_cossim_float, _ = pearsonr(data_eval_dict[label_str], cossim_list)
    spearman_cossim_float, _ = spearmanr(data_eval_dict[label_str], cossim_list)
    
    print(f"{pearson_cossim_float = }")
    print(f"{spearman_cossim_float = }")
    
    filepath_output_path = Path(filepath_output_str)
    if not os.path.isfile(filepath_output_path):
        with filepath_output_path.open(mode = "w") as fp:
            fp.write("")
    
    with open(filepath_output_path, "a") as fp:
        fp.write(f"{type_str}\n")
        fp.write(f"pearson_cossim: {pearson_cossim_float}\n")
        fp.write(f"spearman_cossim: {spearman_cossim_float}\n")
    
    return pearson_cossim_float, spearman_cossim_float


In [ ]:
### Load the validation dataset
import json

filepath_dataset_eval_str = "../../datasets/jsts-v1.1/valid-v1.1.json"

label_str = "label"
sentence1_str = "sentence1"
sentence2_str = "sentence2"


data_eval_dict = dict()
labels_list = list()
sentences1_list = list()
sentences2_list = list()

with open(filepath_dataset_eval_str) as fp:
    for line in fp:
        line_dict = json.loads(line)
        labels_list.append(line_dict[label_str])
        sentences1_list.append(line_dict[sentence1_str])
        sentences2_list.append(line_dict[sentence2_str])

data_eval_dict[label_str] = labels_list
data_eval_dict[sentence1_str] = sentences1_list
data_eval_dict[sentence2_str] = sentences2_list

# print(data_eval_dict)

In [ ]:
### Make an onnxed model
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer


# load transformers and convert to onnx
model_onnx = ORTModelForFeatureExtraction.from_pretrained(model_name_str, 
                                                          from_transformers = True)
tokenizer = AutoTokenizer.from_pretrained(model_name_str)

# save onnx checkpoint and tokenizer
model_onnx.save_pretrained(modelpath_output_onnx_str)
tokenizer.save_pretrained(modelpath_output_onnx_str)

In [ ]:
### Init a pipeline for onnx
st_onnx_pipeline = SentenceTransformerPipeline(
    model = model_onnx,
    tokenizer = tokenizer,
    device = "cpu"
)
# run inference
embedding_test = st_onnx_pipeline("レンガの建物の前を、乳母車を押した女性が歩いています。")
embeddings_test_list = st_onnx_pipeline(["レンガの建物の前を、乳母車を押した女性が歩いています。", "山の上に顔の白い牛が2頭います。"])

# print an excerpt from the sentence embedding
print(embedding_test[0][:5])
# print(embeddings_test_list)
# print(embeddings_test_list[1][0])
print(embeddings_test_list[1][0][:5])

In [ ]:
### Evaluate the onnxed model
pearson_cossim_float, spearman_cossim_float = evaluate_model(
    st_pipeline = st_onnx_pipeline,
    data_eval_dict = data_eval_dict,
    type_str = "<onnxed>",
    filepath_output_str = filepath_output_eval_jsts_str
)

In [ ]:
### izumil_jppost-cp-e1-b512_jsts-100-cp-e3-b1_onnx-optim
## no normalization
# pearson_cossim_float = 0.851494646603602
# spearman_cossim_float = 0.7997368522838116
# 
## with normalization
# pearson_cossim_float = 0.851494646603602
# spearman_cossim_float = 0.7997368522838116
# 

In [ ]:
### Optimize the onnxed model
from optimum.onnxruntime import ORTOptimizer
from optimum.onnxruntime.configuration import OptimizationConfig


# create ORTOptimizer and define optimization configuration
optimizer = ORTOptimizer.from_pretrained(model_onnx)
optimization_config = OptimizationConfig(optimization_level = 99) # enable all optimizations

# apply the optimization configuration to the model
optimizer.optimize(save_dir = modelpath_output_optd_str, 
                   optimization_config = optimization_config,)

In [ ]:
### Init a pipeline with the optimized model
from optimum.onnxruntime import ORTModelForFeatureExtraction

# load optimized model
model_optd = ORTModelForFeatureExtraction.from_pretrained(modelpath_output_optd_str, 
                                                          file_name="model_optimized.onnx")

# create pipeline
st_optd_pipeline = SentenceTransformerPipeline(model = model_optd, 
                                               tokenizer = tokenizer, 
                                               device = "cpu")

# run inference
embedding_test = st_optd_pipeline("レンガの建物の前を、乳母車を押した女性が歩いています。")
embeddings_test_list = st_optd_pipeline(["レンガの建物の前を、乳母車を押した女性が歩いています。", "山の上に顔の白い牛が2頭います。"])

# print an excerpt from the sentence embedding
print(embedding_test[0][:5])
print(embeddings_test_list[1][0][:5])

In [ ]:
### Evaluate the optimized model
pearson_cossim_float, spearman_cossim_float = evaluate_model(st_pipeline = st_optd_pipeline, 
                                                             data_eval_dict = data_eval_dict, 
                                                             type_str = "<optimized>", 
                                                             filepath_output_str = filepath_output_eval_jsts_str)

In [ ]:
### Quantize the onnxed model
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig


# create ORTQuantizer and define quantization configuration
dynamic_quantizer = ORTQuantizer.from_pretrained(model_optd)
dqconfig = AutoQuantizationConfig.avx512_vnni(
    is_static = False,
    per_channel = False
)
# apply the quantization configuration to the model
model_quantized_path = dynamic_quantizer.quantize(
    save_dir = modelpath_output_qtzd_str,
    quantization_config = dqconfig
)

In [ ]:
### Init a pipeline with the quantized model
from optimum.onnxruntime import ORTModelForFeatureExtraction

# load quantized model
model_qtzd = ORTModelForFeatureExtraction.from_pretrained(
    modelpath_output_qtzd_str,
    file_name = "model_optimized_quantized.onnx"
)

# create pipeline
st_qtzd_pipeline = SentenceTransformerPipeline(
    model = model_qtzd,
    tokenizer = tokenizer,
    device = "cpu"
)
# run inference
embedding_test = st_qtzd_pipeline("レンガの建物の前を、乳母車を押した女性が歩いています。")
embeddings_test_list = st_qtzd_pipeline(["レンガの建物の前を、乳母車を押した女性が歩いています。", "山の上に顔の白い牛が2頭います。"])

# print an excerpt from the sentence embedding
print(embedding_test[0][:5])
print(embeddings_test_list[1][0][:5])

In [ ]:
### Evaluate the quantized model
pearson_cossim_float, spearman_cossim_float = evaluate_model(
    st_pipeline = st_qtzd_pipeline,
    data_eval_dict = data_eval_dict,
    type_str = "<quantized>",
    filepath_output_str = filepath_output_eval_jsts_str
)

In [ ]:
### Check the model sizes
import os

size_mb_int = 1024 * 1024

# get model file size
size_model_orig_float = os.path.getsize(model_name_str + "/pytorch_model.bin") / size_mb_int
size_model_onnx_float = os.path.getsize(modelpath_output_onnx_str + "/model.onnx") / size_mb_int
size_model_optd_float = os.path.getsize(modelpath_output_optd_str + "/model_optimized.onnx") / size_mb_int
size_model_qtzd_float = os.path.getsize(modelpath_output_qtzd_str + "/model_optimized_quantized.onnx") / size_mb_int

print(f"orig: {size_model_orig_float:.2f} MB")
print(f"onnx: {size_model_onnx_float:.2f} MB")
print(f"optd: {size_model_optd_float:.2f} MB")
print(f"qtzd: {size_model_qtzd_float:.2f} MB")


with open(filepath_output_eval_jsts_str, "a") as fp:
    fp.write(f"\n<model sizes>\n")
    fp.write(f"orig: {size_model_orig_float:.2f} MB\n")
    fp.write(f"onnx: {size_model_onnx_float:.2f} MB\n")
    fp.write(f"optd: {size_model_optd_float:.2f} MB\n")
    fp.write(f"qtzd: {size_model_qtzd_float:.2f} MB\n")


In [ ]:
# print(st_onnx_pipeline.preprocess("レンガの建物の前を、乳母車を押した女性が歩いています。"))
# print(st_onnx_pipeline.preprocess("レンガの建物の前を、乳母車を押した女性が歩いています。")["input_ids"][0])
# print(len(st_onnx_pipeline.preprocess("レンガの建物の前を、乳母車を押した女性が歩いています。")["input_ids"][0]))

In [ ]:
import time
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LinearRegression

%matplotlib inline
# plt.style.use('dark_background')
plt.style.use('default')


column_charslen_str = 'the_num_of_chars'
column_tokenslen_str = 'the_num_of_tokens'
column_elapsedtime_str = 'elapsed_time'    


def count_tokens(st_pipeline, sentence_str):
    sentence_tokenized = st_onnx_pipeline.preprocess(sentence_str)
    
    return len(sentence_tokenized["input_ids"][0])


def get_elapsed_times(st_pipeline, data_df):
    column_sentence_int = 1
    # column_sentence_str = 'question'
    
    data_df[column_tokenslen_str] = 0
    data_df[column_elapsedtime_str] = 0.
    
    counter = 0
    for row_pdtuple in data_df.itertuples():
        row_index_int = row_pdtuple.Index
        
        sentence_str = row_pdtuple[column_sentence_int]
        num_tokens_int = count_tokens(st_pipeline, sentence_str)
        data_df.loc[row_index_int, column_tokenslen_str] = num_tokens_int
        # print(num_tokens_int)
        
        start_time = time.time()
        embedding = st_pipeline(row_pdtuple[column_sentence_int])
        end_time = time.time()
        
        elapsed_time = end_time - start_time
        
        if counter % 500 == 0:
            print(counter, elapsed_time)
        counter += 1
        
        data_df.loc[row_index_int, column_elapsedtime_str] = elapsed_time
    
    return data_df


def draw_graph(modeltype_str, data_df, column_str, lr_max_float, color_plot_str='#086972', color_lr_str='#DF5E88'):
    # Set coordinates
    x_max_float = 0.0
    if column_str == column_charslen_str:
        x_max_float = 920.0
    elif column_str == column_tokenslen_str:
        x_max_float = 530.0
    
    x_describe_df = data_df[column_str].describe()
    y_describe_df = data_df[column_elapsedtime_str].describe()
    x_min_float = x_max_float / 36
    x_mid_float = x_max_float / 2
    x_indent_float = x_max_float / 18
    y_incrm_float = y_max_float / 30
    y_title_float = y_max_float - y_incrm_float
    y_stats_float = y_max_float / 3
    y_lr_float = y_max_float / 2
    
    
    # Format data
    x_sr = data_df[column_str].values
    y_sr = data_df[column_elapsedtime_str].values
    
    data_lr_df = data_df[data_df[column_elapsedtime_str] <= lr_max_float]
    x_lr_df = data_lr_df[[column_str]].values
    y_lr_sr = data_lr_df[column_elapsedtime_str].values

    
    # Linear Regression
    lr = LinearRegression()
    lr.fit(x_lr_df, y_lr_sr)
    
    coef_float = lr.coef_[0]
    intercept_float = lr.intercept_
    
    # Plot
    plt.clf() # Clear the plot
    elapsed_snsplot = sns.jointplot(x=x_sr, y=y_sr, xlim=(0, x_max_float), ylim=(0, y_max_float), color=color_plot_str)
    plt.plot(x_lr_df, lr.predict(x_lr_df), color=color_lr_str)
    
    # Print titles
    plt.text(x_min_float, y_title_float, f'#{dataset_name.upper()}', horizontalalignment='left')
    plt.text(x_min_float, y_title_float - y_incrm_float, f'#{modeltype_str}', horizontalalignment='left')
    
    # Print the coefficient and the intercept of Linear Regression
    plt.text(x_mid_float, y_lr_float, '<Linear Regression>', horizontalalignment='left')
    plt.text(x_mid_float + x_indent_float, y_lr_float - y_incrm_float * 1, 'coef', horizontalalignment='left')
    plt.text(x_max_float, y_lr_float - y_incrm_float * 1, '%.8f' % coef_float, horizontalalignment='right')
    plt.text(x_mid_float + x_indent_float, y_lr_float - y_incrm_float * 2, 'intercept', horizontalalignment='left')
    plt.text(x_max_float, y_lr_float - y_incrm_float * 2, '%.8f' % intercept_float, horizontalalignment='right')
    
    # Print stats of x
    plt.text(x_min_float, y_stats_float + y_incrm_float * 0.5, f'<{column_str}>', horizontalalignment='left')
    
    for i, key in enumerate(list(x_describe_df.index)):
        plt.text(x_min_float + x_indent_float, y_stats_float - y_incrm_float * (i + 1), key, horizontalalignment='left')
        plt.text(x_mid_float, y_stats_float - y_incrm_float * (i + 1), '%.8f' % x_describe_df[key], horizontalalignment='right')
        
    # Print stats of y
    plt.text(x_mid_float, y_stats_float + y_incrm_float * 0.5, f'<{column_elapsedtime_str}>', horizontalalignment='left')
    
    for i, key in enumerate(list(x_describe_df.index)):
        plt.text(x_mid_float + x_indent_float, y_stats_float - y_incrm_float * (i + 1), key, horizontalalignment='left')
        plt.text(x_max_float, y_stats_float - y_incrm_float * (i + 1), '%.8f' % y_describe_df[key], horizontalalignment='right')
        

    plt.show()
    
    # Save the plot
    x_type_str = column_str.replace('the_num_of', '')
    filepath_output = 'elapsed_time/elapsed_time_' + modeltype_str + x_type_str + '_' + dataset_name + '_h' + str(y_max_float) + '.png'
    elapsed_snsplot.figure.savefig(filepath_output)



In [ ]:
# Load the datasets
data_df = pd.read_csv(path_dataset_str, index_col=0)
data_df.head()

In [ ]:
# pd.set_option("display.max_rows", 80)
lr_max_onnx_float = 1.0
lr_max_optd_float = 1.0
lr_max_qtzd_float = 1.0

# data_onnx_outl_df = data_onnx_df[data_onnx_df[column_elapsedtime_str] > lr_max_onnx_float]
# print(data_onnx_outl_df.shape[0])
# data_onnx_outl_df

# data_optd_outl_df = data_optd_df[data_optd_df[column_elapsedtime_str] > lr_max_optd_float]
# print(data_optd_outl_df.shape[0])
# data_optd_outl_df

# data_qtzd_outl_df = data_qtzd_df[data_qtzd_df[column_elapsedtime_str] > lr_max_qtzd_float]
# print(data_qtzd_outl_df.shape[0])
# data_qtzd_outl_df


In [ ]:
# Measure elapsed time with the onnxed model
data_onnx_df = data_df.copy()
data_onnx_df = get_elapsed_times(st_onnx_pipeline, data_onnx_df)
data_onnx_df.head()

# Draw the graphs
draw_graph(modeltype_onnx_str, data_onnx_df, column_charslen_str, lr_max_onnx_float, color_plot_str='#086972')
draw_graph(modeltype_onnx_str, data_onnx_df, column_tokenslen_str, lr_max_onnx_float, color_plot_str='#277BC0')

In [ ]:
# Measure elapsed time with the optimized model
data_optd_df = data_df.copy()
data_optd_df = get_elapsed_times(st_optd_pipeline, data_optd_df)
data_optd_df.head()

# Draw the graphs
draw_graph(modeltype_optd_str, data_optd_df, column_charslen_str, lr_max_optd_float, color_plot_str='#086972')
draw_graph(modeltype_optd_str, data_optd_df, column_tokenslen_str, lr_max_optd_float, color_plot_str='#277BC0')

In [ ]:
# Measure elapsed time with the quantized model
data_qtzd_df = data_df.copy()
data_qtzd_df = get_elapsed_times(st_qtzd_pipeline, data_qtzd_df)
data_qtzd_df.head()

# Draw the graphs
draw_graph(modeltype_qtzd_str, data_qtzd_df, column_charslen_str, lr_max_qtzd_float, color_plot_str='#086972')
draw_graph(modeltype_qtzd_str, data_qtzd_df, column_tokenslen_str, lr_max_qtzd_float, color_plot_str='#277BC0')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sentence_transformers import models, SentenceTransformer, util

%matplotlib inline
# plt.style.use('dark_background')
plt.style.use('default')

COSSIM_NAME = 'cos_sim'


def set_chimei_categories_list():
    chimei_categories_list = list()
    chimei_categories_list.append('00_prefectures')
    chimei_categories_list.append('01_shi')
    chimei_categories_list.append('02_ku')
    chimei_categories_list.append('03_gun')
    chimei_categories_list.append('04_cho')
    chimei_categories_list.append('05_son')
    
    return chimei_categories_list


def set_filepaths_input_dict(chimei_categories_list):
    filepaths_input_base = '../../datasets/jppost/local_govt_lists/'
    filepaths_input_dict = dict()

    for chimei_category in chimei_categories_list:
        filepaths_input = filepaths_input_base + chimei_category + '_list.txt'
        filepaths_input_dict[chimei_category] = filepaths_input
    
    return filepaths_input_dict
        

def set_chimeis_lists_dict(filepaths_input_dict):
    chimeis_lists_dict = dict()
    
    for chimei_category, filepath_input in filepaths_input_dict.items():
        chimeis_list = list()
        
        with open(filepath_input) as fp:
            for line in fp:
                chimeis_list.append(line.rstrip())
        
        chimeis_lists_dict[chimei_category] = chimeis_list
    
    return chimeis_lists_dict


def embed_chimei(st_pipeline, chimeis_lists_dict):
    embeddings_dicts_dict = dict()
    
    for chimei_category, chimeis_list in chimeis_lists_dict.items():
        embeddings_dict = dict()
        
        for chimei in chimeis_list:
            chimei_embedding = st_pipeline(chimei)[0]
            embeddings_dict[chimei] = chimei_embedding
        
        embeddings_dicts_dict[chimei_category] = embeddings_dict
    
    return embeddings_dicts_dict


def set_filepaths_output_dict(chimei_categories_list, directory, prefix, modeltype_str, ext):
    filepaths_output_dict = dict()

    for chimei_category in chimei_categories_list:
        filepaths_output = directory + prefix + modeltype_str + '_' + chimei_category + ext
        filepaths_output_dict[chimei_category] = filepaths_output
    
    return filepaths_output_dict


def create_cossim_matrix_dfs_dict(embeddings_dicts_dict):
    cossim_matrix_dfs_dict = dict()
    covered_chimeis_list = list()
    
    for chimei_category, embeddings_dict in embeddings_dicts_dict.items():
        chimeis_list = list(embeddings_dict.keys())
        cossim_matrix_df = pd.DataFrame(index=chimeis_list, columns=chimeis_list)
        
        for column in chimeis_list:
            covered_chimeis_list.append(column)
            
            for row in chimeis_list:
                if row not in covered_chimeis_list:
                    cossim_tensor = util.cos_sim(embeddings_dict[row], embeddings_dict[column])[0][0]
                    cossim_matrix_df.loc[row, column] = cossim_tensor.tolist()
            
            cossim_matrix_df[column] = pd.to_numeric(cossim_matrix_df[column], errors='coerce')
        
        cossim_matrix_df = cossim_matrix_df.transpose()
        cossim_matrix_dfs_dict[chimei_category] = cossim_matrix_df
        
    return cossim_matrix_dfs_dict


def melt_cossim_dfs_dict(cossim_matrix_dfs_dict):
    cossim_melted_dfs_dict = dict()
    
    for chimei_category, cossim_matrix_df in cossim_matrix_dfs_dict.items():
        
        cossim_melted_df = pd.melt(
            cossim_matrix_df.reset_index(),
            id_vars='index',
            var_name='local_govt_name2',
            value_name=COSSIM_NAME
        )
        cossim_melted_df = cossim_melted_df.rename(columns={'index': 'local_govt_name1'})
        
        cossim_melted_df = cossim_melted_df.dropna()
        
        # Sort by cosine similarity and reset the index
        cossim_melted_df = cossim_melted_df.sort_values(COSSIM_NAME, ascending=False)
        cossim_melted_df = cossim_melted_df.reset_index()
        cossim_melted_df = cossim_melted_df.drop(columns='index')
        
        cossim_melted_dfs_dict[chimei_category] = cossim_melted_df
        
    return cossim_melted_dfs_dict


def plot_and_save_cossim_hists(cossim_dfs_dict, filepaths_output_cossim_hist_dict, modeltype_str):
    
    for chimei_category, cossim_df in cossim_dfs_dict.items():
        plt.clf() # Clear the plot
        cossim_hist_plot = sns.histplot(cossim_df[COSSIM_NAME], kde=True, binrange=(-1,1))
        
        # Get the maximun y-coordinate
        ys_np = cossim_hist_plot.lines[0].get_ydata()
        y_max_float = np.max(ys_np)
        
        # Get the stats
        cossim_describe_df = cossim_df[COSSIM_NAME].describe()
        
        # Add the stats table on the plot
        plt.text(-0.99, y_max_float, modeltype_str, horizontalalignment='left')
        for i, key in enumerate(list(cossim_describe_df.index)):
            plt.text(-0.99, y_max_float - (i + 1) * y_max_float / 16, key, horizontalalignment='left')
            plt.text(-0.19, y_max_float - (i + 1) * y_max_float / 16, '%.8f' % cossim_describe_df[key], horizontalalignment='right')
        
        # Draw lines for the stats
        color_quarters = 'orange'
        color_standard = 'gray'
        linewidth_float = 0.5
        zorder_int = -1
        
        std_lower_float = cossim_describe_df['mean'] - cossim_describe_df['std']
        std_upper_float = cossim_describe_df['mean'] + cossim_describe_df['std']
        
        plt.axvline(cossim_describe_df['max'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['min'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['50%'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['25%'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['75%'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['mean'], color=color_standard, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(std_lower_float, color=color_standard, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(std_upper_float, color=color_standard, linewidth=linewidth_float, zorder=zorder_int)

        
        fig_hist = cossim_hist_plot.get_figure()
        fig_hist.savefig(filepaths_output_cossim_hist_dict[chimei_category])

    return None


In [ ]:
# Create a chimei category list.
chimei_categories_list = set_chimei_categories_list()

# Create a dictionary of the paths of the input files by chimei category.
filepaths_input_dict = set_filepaths_input_dict(chimei_categories_list)

# Create a dictionary of chimei lists by chimei category.
chimeis_lists_dict = set_chimeis_lists_dict(filepaths_input_dict)

In [ ]:
### onnx
# Embed chimei lists via the model.
embeddings_dicts_dict = embed_chimei(st_onnx_pipeline, chimeis_lists_dict)

# Create a dictionary of the dataframes whose values are the cosine similarities of the chimei pairs by chimei category.
cossim_matrix_dfs_dict = create_cossim_matrix_dfs_dict(embeddings_dicts_dict)

# Create a dictionary of the dataframes created with cossim matrices melted by chimei category.
cossim_melted_dfs_dict = melt_cossim_dfs_dict(cossim_matrix_dfs_dict)

# Create a dictionary of the paths of the files to save the histgrams of the melted cossim matrices as by chimei category.
filepaths_output_cossim_mltd_hist_dict = set_filepaths_output_dict(
    chimei_categories_list=chimei_categories_list,
    directory='./cossim_1_melted_1_hist/',
    prefix='cossim_melted_hist2_',
    modeltype_str=modeltype_onnx_str,
    ext='.png'
)
# Plot and save histgrams of melted cossim matrices as files by chimei category.
plot_and_save_cossim_hists(cossim_melted_dfs_dict, filepaths_output_cossim_mltd_hist_dict, modeltype_onnx_str)


In [ ]:
### optd
# Embed chimei lists via the model.
embeddings_dicts_dict = embed_chimei(st_optd_pipeline, chimeis_lists_dict)

# Create a dictionary of the dataframes whose values are the cosine similarities of the chimei pairs by chimei category.
cossim_matrix_dfs_dict = create_cossim_matrix_dfs_dict(embeddings_dicts_dict)

# Create a dictionary of the dataframes created with cossim matrices melted by chimei category.
cossim_melted_dfs_dict = melt_cossim_dfs_dict(cossim_matrix_dfs_dict)

# Create a dictionary of the paths of the files to save the histgrams of the melted cossim matrices as by chimei category.
filepaths_output_cossim_mltd_hist_dict = set_filepaths_output_dict(
    chimei_categories_list=chimei_categories_list,
    directory='./cossim_1_melted_1_hist/',
    prefix='cossim_melted_hist2_',
    modeltype_str=modeltype_optd_str,
    ext='.png'
)
# Plot and save histgrams of melted cossim matrices as files by chimei category.
plot_and_save_cossim_hists(cossim_melted_dfs_dict, filepaths_output_cossim_mltd_hist_dict, modeltype_optd_str)


In [ ]:
### qtzd
# Embed chimei lists via the model.
embeddings_qtzd_dicts_dict = embed_chimei(st_qtzd_pipeline, chimeis_lists_dict)

# Create a dictionary of the dataframes whose values are the cosine similarities of the chimei pairs by chimei category.
cossim_matrix_qtzd_dfs_dict = create_cossim_matrix_dfs_dict(embeddings_qtzd_dicts_dict)

# Create a dictionary of the dataframes created with cossim matrices melted by chimei category.
cossim_melted_qtzd_dfs_dict = melt_cossim_dfs_dict(cossim_matrix_qtzd_dfs_dict)

# Create a dictionary of the paths of the files to save the histgrams of the melted cossim matrices as by chimei category.
filepaths_output_cossim_mltd_hist_dict = set_filepaths_output_dict(
    chimei_categories_list=chimei_categories_list,
    directory='./cossim_1_melted_1_hist/',
    prefix='cossim_melted_hist2_',
    modeltype_str=modeltype_qtzd_str,
    ext='.png'
)
# Plot and save histgrams of melted cossim matrices as files by chimei category.
plot_and_save_cossim_hists(cossim_melted_qtzd_dfs_dict, filepaths_output_cossim_mltd_hist_dict, modeltype_qtzd_str)
